# Placement Success Prediction

This notebook generates the explainability outputs for the trained model:

- SHAP summary plot
- feature importance table
- LIME local explanation


In [ ]:
from pathlib import Path
import sys

import joblib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_ingestion import ingest_data
from preprocessing import split_features_target
from explainability import (
    build_transformed_frame,
    ensure_reports_dir,
    generate_lime_explanation,
    generate_shap_summary,
    save_feature_importance,
)


In [ ]:
BASE_DIR = ROOT
MODEL_DIR = BASE_DIR / 'models'
MODEL_PATH = MODEL_DIR / 'xgb_model.pkl'
SCALER_PATH = MODEL_DIR / 'scaler.pkl'
ENCODER_PATH = MODEL_DIR / 'encoder.pkl'

NUMERIC_FEATURES = [
    'Heart Rate (bpm)',
    'SpO2 Level (%)',
    'Systolic Blood Pressure (mmHg)',
    'Diastolic Blood Pressure (mmHg)',
    'Body Temperature (°C)',
    'Data Accuracy (%)',
    'Pulse Pressure (mmHg)',
    'Mean Arterial Pressure (mmHg)',
    'Is Recent Fall Detected',
    'Is Fever',
    'Low SpO2 Flag',
]
CATEGORICAL_FEATURES = ['Recent Fall']

artifact = joblib.load(MODEL_PATH)
model = artifact['model'] if isinstance(artifact, dict) else artifact
scaler = joblib.load(SCALER_PATH)
encoder = joblib.load(ENCODER_PATH)

df = ingest_data()
X, y = split_features_target(df)
X.head()


In [ ]:
transformed = build_transformed_frame(X, scaler, encoder, NUMERIC_FEATURES, CATEGORICAL_FEATURES)
reports_dir = ensure_reports_dir()
feature_names = list(transformed.columns)

feature_importance_path = save_feature_importance(
    model,
    reports_dir / 'feature_importance.csv',
    feature_names=feature_names,
)
shap_path = generate_shap_summary(model, transformed, reports_dir / 'shap_summary.png')
lime_path = generate_lime_explanation(model, transformed, 0, reports_dir / 'lime_explanation.html')

print('Saved feature importance to', feature_importance_path)
print('Saved SHAP summary to', shap_path)
print('Saved LIME explanation to', lime_path)


In [ ]:
pd.read_csv(feature_importance_path).head(10)
